Summary: on benchmark le dataloader

In [1]:
from retinotopy import *
welcome()

-----------------------------------------------------------------------------------------
On date 2025-01-05, Running learning on host obiwan.local with device mps, pytorch==2.5.1
-----------------------------------------------------------------------------------------
Welcome on macOS-15.2-arm64-arm-64bit


# Loading legacy images

In [2]:
args = Params()
data_set_type = 'full'
args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images
args.folders = ['train', 'val'] # type of images to use
args

Params(datetag='2025-01-05', loader='data/Imagenet_urls_ILSVRC_2016.json', annotations_animal='data/Animal10k_annotations.json', annotations_val='data/LOC_val_solution_with_sizes.csv', folders=['train', 'val'], tasks=['animal', 'dog', 'cat', 'bird'], image_size=224, num_epochs=2, n_train_stop=0, seed=1998, batch_size=75, batch_size_val=75, lr=0.00015, momentum=0.06, beta2=0, rs_min=0.0, rs_max=-5.0, do_polar=True, do_raw=False, do_translate=False, do_resize=True, do_mask=True, do_scratch=False, do_rotation=False, resolution=(11, 11), size_ratio=0.1, do_saccade=False, do_zoom=False, method='valid', saccade_type='multi', normalize=True, verbose=False)

In [3]:
%%timeit -n1
args.folders = ['train', 'val'] # type of images to use
dataloaders = datasets_transforms(args)
len(dataloaders['train']), len(dataloaders['train'].dataset)

Loaded 1281167 images under train
Loaded 50000 images under val
Loaded 1281167 images under train
Loaded 50000 images under val
Loaded 1281167 images under train
Loaded 50000 images under val
Loaded 1281167 images under train
Loaded 50000 images under val
Loaded 1281167 images under train
Loaded 50000 images under val
Loaded 1281167 images under train
Loaded 50000 images under val
Loaded 1281167 images under train
Loaded 50000 images under val
2.93 s ± 983 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [4]:
%%timeit -n1
args.folders = ['val'] # type of images to use
dataloaders = datasets_transforms(args)
len(dataloaders['val']), len(dataloaders['val'].dataset)

Loaded 50000 images under val
Loaded 50000 images under val
Loaded 50000 images under val
Loaded 50000 images under val
Loaded 50000 images under val
Loaded 50000 images under val
Loaded 50000 images under val
91.4 ms ± 11 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


Benchmarking different methods for the dataloader:

In [ ]:
for num_workers_ in [0, 1, 2, 5, 8, 16, 32]:
    for batch_size_ in [1, 4, 16, 32, 128, 256, 512]:
        for pin_memory_ in [True, False]:
            args = Params()
            args.batch_size = batch_size_
            args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images
            args.folders = ['val'] # type of images to use            
        
            dataloaders = datasets_transforms(args, pin_memory=pin_memory_, num_workers=num_workers_, verbose=False)
            tic = time.time()
            i_image, i_image_max = 0, 4096
            for i_step, (images, labels) in enumerate(dataloaders['val']):
                    images, labels = images.to(device), labels.to(device)
                    i_image += len(images)
                    if i_image > i_image_max:
                        break

            toc = time.time()
            print(f'{pin_memory_=} \t {num_workers_=} \t {batch_size_=} \t Loading time for {i_image_max} images \t {toc-tic:.1f} s')  

pin_memory_=True 	 num_workers_=0 	 batch_size_=1 	 Loading time for 4096 images 	 17.2 s
pin_memory_=False 	 num_workers_=0 	 batch_size_=1 	 Loading time for 4096 images 	 17.0 s
